# CUDA English Video Subtitles

This notebook writes plain `.srt` subtitle files beside videos that do not already have subtitles. It uses `distil-large-v3` with CUDA and English transcription. It does not diarize speakers, upload media, or mux subtitles into videos.

Prerequisites: a working NVIDIA driver, Python 3.10 or 3.11, and internet access for package/model downloads. Run the install cell once, restart the kernel, then run the remaining cells. NVIDIA's CUDA Toolkit is not required separately; the Python CUDA runtime libraries are installed below.

The current `Resources` videos with supplied original captions are explicitly mapped and skipped. Other same-stem subtitle files (`.srt`, `.vtt`, `.ass`, `.ssa`, `.sub`, `.ttml`, `.sbv`) are also treated as already subtitled.

In [ ]:
%pip install -U faster-whisper nvidia-cublas-cu12 "nvidia-cudnn-cu12>=9,<10"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Make pip-installed NVIDIA runtime DLLs discoverable by CTranslate2 on Windows.
_dll_handles = []
site_packages = Path(sys.prefix) / "Lib" / "site-packages"
if os.name == "nt":
    for dll_dir in (site_packages / "nvidia").glob("*/bin"):
        if dll_dir.is_dir():
            _dll_handles.append(os.add_dll_directory(str(dll_dir)))
            os.environ["PATH"] = str(dll_dir) + os.pathsep + os.environ.get("PATH", "")

nvidia_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if nvidia_smi.returncode != 0:
    raise RuntimeError("NVIDIA driver/GPU not detected. Install or update the NVIDIA driver, then restart VS Code.")
print("NVIDIA GPU detected:")
print("\n".join(nvidia_smi.stdout.splitlines()[:5]))

import ctranslate2

cuda_types = ctranslate2.get_supported_compute_types("cuda")
print("CTranslate2 CUDA compute types:", sorted(cuda_types))
if "float16" not in cuda_types:
    raise RuntimeError("CUDA float16 is unavailable. Check the NVIDIA driver and CUDA/cuDNN runtime installation.")

In [ ]:
INPUT_DIR = Path("Resources")
VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}
SUBTITLE_EXTENSIONS = {".srt", ".vtt", ".ass", ".ssa", ".sub", ".ttml", ".sbv"}
MODEL_NAME = "distil-large-v3"
LANGUAGE = "en"

# These course videos already have captions whose filenames do not match the videos.
EXISTING_CAPTIONS = {
    "Stakeholder management [9thejifzSZM].mp4": "stakeholder - original captions.srt",
    "Project leader [4DKGxtKx84E].mp4": "leader - original captions.srt",
}

def find_subtitle_sidecar(video_path: Path):
    mapped_name = EXISTING_CAPTIONS.get(video_path.name)
    if mapped_name and (video_path.parent / mapped_name).is_file():
        return video_path.parent / mapped_name

    video_stem = video_path.stem.casefold()
    for sibling in video_path.parent.iterdir():
        if sibling.is_file() and sibling.stem.casefold() == video_stem:
            if sibling.suffix.casefold() in SUBTITLE_EXTENSIONS:
                return sibling
    return None

videos = sorted(
    (path for path in INPUT_DIR.rglob("*") if path.is_file() and path.suffix.casefold() in VIDEO_EXTENSIONS),
    key=lambda path: str(path).casefold(),
)
video_queue = [path for path in videos if find_subtitle_sidecar(path) is None]
skipped_videos = [path for path in videos if find_subtitle_sidecar(path) is not None]

print(f"Videos found: {len(videos)}")
print(f"Already subtitled (will skip): {len(skipped_videos)}")
print(f"Videos queued for English subtitles: {len(video_queue)}")
for video_path in video_queue:
    print(" -", video_path)

## CUDA English Model

`distil-large-v3` is an English-focused Whisper model. Voice activity detection (VAD) filters silent regions before recognition; a 1-second minimum silence interval is used below.

In [ ]:
from faster_whisper import WhisperModel

model = None
if video_queue:
    model = WhisperModel(
        MODEL_NAME,
        device="cuda",
        compute_type="float16",
    )
    print(f"Loaded {MODEL_NAME} on CUDA with float16.")
else:
    print("Nothing to transcribe; no model was loaded.")

In [ ]:
import time


def format_srt_timestamp(seconds: float) -> str:
    milliseconds = max(0, round(seconds * 1000))
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    seconds_part, milliseconds_part = divmod(remainder, 1000)
    return f"{hours:02}:{minutes:02}:{seconds_part:02},{milliseconds_part:03}"


def write_srt(segments, destination: Path) -> None:
    lines = []
    for number, segment in enumerate(segments, start=1):
        text = segment.text.strip()
        if not text:
            continue
        lines.extend([
            str(number),
            f"{format_srt_timestamp(segment.start)} --> {format_srt_timestamp(segment.end)}",
            text,
            "",
        ])

    if not lines:
        raise RuntimeError("No speech was detected; no subtitle file was written.")

    temporary_path = destination.with_name(destination.name + ".tmp")
    try:
        temporary_path.write_text("\n".join(lines), encoding="utf-8-sig")
        temporary_path.replace(destination)
    finally:
        temporary_path.unlink(missing_ok=True)


processed = []
skipped_during_run = []
failed = []

for index, video_path in enumerate(video_queue, start=1):
    existing_subtitle = find_subtitle_sidecar(video_path)
    if existing_subtitle is not None:
        print(f"[{index}/{len(video_queue)}] Skip: subtitle exists ({existing_subtitle.name})")
        skipped_during_run.append(video_path)
        continue

    subtitle_path = video_path.with_suffix(".srt")
    started_at = time.perf_counter()
    print(f"[{index}/{len(video_queue)}] Transcribing: {video_path.name}")

    try:
        segments, info = model.transcribe(
            str(video_path),
            language=LANGUAGE,
            beam_size=5,
            vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 1000},
        )
        segments = list(segments)
        write_srt(segments, subtitle_path)
        elapsed = time.perf_counter() - started_at
        processed.append(video_path)
        print(f"Saved {subtitle_path.name} ({info.language}, {elapsed:.1f}s)")
    except Exception as error:
        failed.append((video_path, str(error)))
        print(f"Failed: {video_path.name}: {error}")

print("\nDone.")
print(f"Subtitles written: {len(processed)}")
print(f"Skipped during run: {len(skipped_during_run)}")
print(f"Failed: {len(failed)}")